# TQ Sleeve Allocation Framework — α Engine

Milestone v2.0. PM-readable scorecard of options-overlay sleeve attractiveness across SPX/QQQ (and optionally XIU/XSP), backed by sleeve P&L backtests.

v1.0 legacy: `hmm.ipynb` (untouched).

## Section 0 — Setup & Config

In [ ]:
import pathlib
import warnings
import datetime as dt

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal

In [ ]:
UNDERLYINGS = {
    "SPX":  "SPX Index",
    "QQQ":  "QQQ US Equity",
    "XIU":  "XIU CN Equity",
    "XSP":  "XSP CN Equity",
}
REQUIRED_UNDERLYINGS = ("SPX", "QQQ")

IV_FIELDS = {
    "iv30_atm":   "30DAY_IMPVOL_100.0%MNY_DF",
    "iv30_90mny": "30DAY_IMPVOL_90.0%MNY_DF",
    "iv90_atm":   "90DAY_IMPVOL_100.0%MNY_DF",
}
PRICE_FIELD = "PX_LAST"

CROSS_ASSET = {
    "vix":     ("VIX Index",     "PX_LAST"),
    "vvix":    ("VVIX Index",    "PX_LAST"),
    "rf_rate": ("USGG3M Index",  "PX_LAST"),
}

START_DATE = "2005-01-01"
END_DATE   = dt.date.today().isoformat()

In [ ]:
CACHE_DIR = pathlib.Path.home() / "sleeve_alpha_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def nyse_index(start, end):
    sched = mcal.get_calendar("NYSE").schedule(start_date=start, end_date=end)
    return pd.DatetimeIndex(sched.index).normalize()

NYSE_INDEX = nyse_index(START_DATE, END_DATE)

## Section 1 — Data Layer

### 1.1 Cache wrapper

In [ ]:
def _cache_path(ticker, field, start, end):
    safe_t = ticker.replace(" ", "_").replace("/", "_")
    safe_f = field.replace("%", "pct").replace(".", "p").replace(" ", "_")
    return CACHE_DIR / f"{safe_t}__{safe_f}__{start}__{end}.parquet"

def _is_stale(path, max_age_trading_days=1):
    if not path.exists():
        return True
    mtime = dt.datetime.fromtimestamp(path.stat().st_mtime).date()
    today = dt.date.today()
    cal = mcal.get_calendar("NYSE")
    sched = cal.schedule(start_date=mtime, end_date=today)
    trading_days_since = max(0, len(sched) - 1)
    return trading_days_since > max_age_trading_days

def bdh_cached(ticker, field, start, end, force_refresh=False):
    path = _cache_path(ticker, field, start, end)
    if not force_refresh and not _is_stale(path):
        return pd.read_parquet(path)
    try:
        df = con.bdh(ticker, field, start_date=start, end_date=end)
    except Exception as e:
        if path.exists():
            warnings.warn(f"con.bdh failed for {ticker}/{field} ({e}); serving stale cache")
            return pd.read_parquet(path)
        raise
    if df is None or len(df) == 0:
        if path.exists():
            path.unlink()
        return pd.DataFrame()
    df.to_parquet(path)
    return df

### 1.2 Raw pulls — price + IV per underlying

Probes SPX, QQQ (required) and XIU, XSP (best-effort) for `PX_LAST` plus the three
IV fields. Empty responses are logged and the cell stays green; downstream panel
construction branches on column membership, not on assumptions about what landed.

In [ ]:
raw_pulls = {u: {} for u in UNDERLYINGS}

def _probe(label, ticker, field, start, end):
    df = bdh_cached(ticker, field, start, end)
    if df is None or len(df) == 0:
        print(f"  [deferred] {label} / {field}: empty response from con.bdh")
        return pd.DataFrame()
    n = len(df)
    first = df.index.min().date() if hasattr(df.index.min(), "date") else df.index.min()
    last  = df.index.max().date() if hasattr(df.index.max(), "date") else df.index.max()
    print(f"  [landed]   {label} / {field}: n={n}, {first} → {last}")
    return df

for label, ticker in UNDERLYINGS.items():
    print(f"{label} ({ticker}):")
    raw_pulls[label]["price"] = _probe(label, ticker, PRICE_FIELD, START_DATE, END_DATE)
    for fkey, fname in IV_FIELDS.items():
        raw_pulls[label][fkey] = _probe(label, ticker, fname, START_DATE, END_DATE)

missing_required = [
    u for u in REQUIRED_UNDERLYINGS
    if len(raw_pulls[u].get("price", pd.DataFrame())) == 0
    or len(raw_pulls[u].get("iv30_atm", pd.DataFrame())) == 0
]
if missing_required:
    raise RuntimeError(f"Required underlyings missing price or IV: {missing_required}")

### 1.3 Raw pulls — cross-asset (VIX, VVIX, risk-free)

In [ ]:
raw_cross = {}
print("Cross-asset:")
for name, (ticker, field) in CROSS_ASSET.items():
    raw_cross[name] = _probe(name, ticker, field, START_DATE, END_DATE)

if len(raw_cross["vix"]) == 0:
    raise RuntimeError("VIX pull returned empty — required for fragility composite")
if len(raw_cross["rf_rate"]) == 0:
    warnings.warn(
        "USGG3M Index PX_LAST returned empty; downstream BS pricing will need an alternate "
        "risk-free field. Phase 1 ships with rf_rate empty."
    )

### 1.4 Panel assembly

Wide panels on the NYSE trading-day index. `iv90_panel` (90D ATM IV from D-03) is
kept as an intermediate for QA / future term-structure work but is not one of the
three handoff panels. Per D-04, panels are truncated to the shortest IV history
across underlyings to keep signals dense.

Naming: `iv90mny` is the 90%-moneyness 30D IV (skew leg per D-06). `iv90_panel`
is the 90-day ATM IV probe per D-03. `skew_panel = iv90mny - iv_panel`.

In [ ]:
def _to_series(df, name):
    if df is None or len(df) == 0:
        return None
    s = df.squeeze()
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s.index = pd.DatetimeIndex(s.index).normalize()
    s.name = name
    return s

def _build_panel(field_key, underlyings_with_data):
    cols = {}
    for u in underlyings_with_data:
        s = _to_series(raw_pulls[u][field_key], u)
        if s is not None:
            cols[u] = s.reindex(NYSE_INDEX)
    return pd.DataFrame(cols, index=NYSE_INDEX)

landed = [
    u for u in UNDERLYINGS
    if len(raw_pulls[u].get("price", pd.DataFrame())) > 0
    and len(raw_pulls[u].get("iv30_atm", pd.DataFrame())) > 0
    and len(raw_pulls[u].get("iv30_90mny", pd.DataFrame())) > 0
]
print(f"Landed underlyings (price + iv30_atm + iv30_90mny all present): {landed}")

prices_panel_full = _build_panel("price",      landed)
iv_panel_full     = _build_panel("iv30_atm",   landed)
iv90mny_full      = _build_panel("iv30_90mny", landed)
iv90_panel_full   = _build_panel(
    "iv90_atm",
    [u for u in landed if len(raw_pulls[u].get("iv90_atm", pd.DataFrame())) > 0],
)

iv_first_dates = []
for u in landed:
    s = iv_panel_full[u].dropna()
    if len(s):
        iv_first_dates.append(s.index.min())
truncate_start = max(iv_first_dates) if iv_first_dates else NYSE_INDEX.min()
print(f"Truncating panels to start={truncate_start.date()} (shortest IV history per D-04)")

mask = NYSE_INDEX >= truncate_start
prices_panel = prices_panel_full.loc[mask].copy()
iv_panel     = iv_panel_full.loc[mask].copy()
iv90mny      = iv90mny_full.loc[mask].copy()
iv90_panel   = iv90_panel_full.loc[mask].copy()

common_skew_cols = [c for c in iv_panel.columns if c in iv90mny.columns]
skew_panel = iv90mny[common_skew_cols] - iv_panel[common_skew_cols]

### 1.5 Cross-asset Series

`vix`, `vvix`, `rf_rate` are kept as standalone date-indexed Series — VIX is not
a price (D-07), and a fourth panel for one column each adds noise. Reindexed to
the same NYSE_INDEX with the same D-04 truncation. `rf_rate` stays in raw % units
(BS pricer in Phase 3 normalizes to decimal).

In [ ]:
def _to_named_series(name):
    df = raw_cross.get(name, pd.DataFrame())
    if df is None or len(df) == 0:
        s = pd.Series(index=NYSE_INDEX, dtype="float64", name=name)
    else:
        s = df.squeeze()
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]
        s.index = pd.DatetimeIndex(s.index).normalize()
        s.name = name
        s = s.reindex(NYSE_INDEX)
    return s.loc[NYSE_INDEX >= truncate_start]

vix     = _to_named_series("vix")
vvix    = _to_named_series("vvix")
rf_rate = _to_named_series("rf_rate")

for s in (vix, vvix, rf_rate):
    assert isinstance(s, pd.Series), f"{s.name} must be a Series, got {type(s)}"
    assert s.index.equals(prices_panel.index), f"{s.name} index does not match panels"

### 1.6 Freshness check

Per series, prints last_bar_date, trading days stale, and a fresh/stale/empty
status. Per D-13, warns (does not raise) when any series is stale > 5 trading
days — refresh on weekends/holidays should not be blocked.


In [ ]:
def _last_bar(s):
    if s is None or len(s) == 0:
        return None
    valid = s.dropna()
    if len(valid) == 0:
        return None
    return valid.index.max()

def _trading_days_stale(last_date):
    if last_date is None:
        return None
    today = pd.Timestamp(dt.date.today())
    if last_date >= today:
        return 0
    sched = mcal.get_calendar("NYSE").schedule(start_date=last_date, end_date=today)
    return max(0, len(sched) - 1)

rows = []
for panel_name, panel in [
    ("prices_panel",  prices_panel),
    ("iv_panel",      iv_panel),
    ("iv90mny",       iv90mny),
    ("iv90_panel",    iv90_panel),
    ("skew_panel",    skew_panel),
]:
    for col in panel.columns:
        s = panel[col]
        lb = _last_bar(s)
        ds = _trading_days_stale(lb)
        if lb is None:
            status = "empty"
        elif ds is not None and ds > 5:
            status = "stale"
        else:
            status = "fresh"
        rows.append({
            "underlying":    col,
            "field":         panel_name,
            "last_bar_date": lb.date() if lb is not None else None,
            "days_stale":    ds,
            "status":        status,
        })

for name, s in [("vix", vix), ("vvix", vvix), ("rf_rate", rf_rate)]:
    lb = _last_bar(s)
    ds = _trading_days_stale(lb)
    if lb is None:
        status = "empty"
    elif ds is not None and ds > 5:
        status = "stale"
    else:
        status = "fresh"
    rows.append({
        "underlying":    "—",
        "field":         name,
        "last_bar_date": lb.date() if lb is not None else None,
        "days_stale":    ds,
        "status":        status,
    })

freshness_table = pd.DataFrame(rows, columns=["underlying", "field", "last_bar_date", "days_stale", "status"])
print(freshness_table.to_string(index=False))

stale_rows = freshness_table[freshness_table["status"] == "stale"]
if len(stale_rows) > 0:
    warnings.warn(
        f"{len(stale_rows)} series stale > 5 trading days. "
        f"Worst: {int(stale_rows['days_stale'].max())} days. "
        "Consider re-running with force_refresh=True."
    )


### 1.7 Panel missingness QA

Per (panel, underlying), prints n, missingness %, and first/last non-NaN dates.
Cross-calendar NaN (TSX holidays in XIU/XSP columns) shows up here as expected
per D-09 — ~5 NaN/yr per Canadian underlying is documented and tolerable.


In [ ]:
qa_rows = []
for panel_name, panel in [
    ("prices_panel",  prices_panel),
    ("iv_panel",      iv_panel),
    ("iv90mny",       iv90mny),
    ("iv90_panel",    iv90_panel),
    ("skew_panel",    skew_panel),
]:
    for col in panel.columns:
        s = panel[col]
        n = len(s)
        n_valid = int(s.notna().sum())
        miss_pct = 0.0 if n == 0 else 100.0 * (n - n_valid) / n
        valid = s.dropna()
        first = valid.index.min().date() if len(valid) else None
        last  = valid.index.max().date() if len(valid) else None
        qa_rows.append({
            "panel":      panel_name,
            "underlying": col,
            "n":          n,
            "n_valid":    n_valid,
            "miss_pct":   round(miss_pct, 2),
            "first":      first,
            "last":       last,
        })

qa_table = pd.DataFrame(qa_rows, columns=["panel", "underlying", "n", "n_valid", "miss_pct", "first", "last"])
print(qa_table.to_string(index=False))
